# Recolly — Voice pipeline evaluation

```
audio ──[ASR: PhoWhisper]──► transcript ──[Parser: Ollama LLM]──► todo_draft
        Tier 1: WER, KR, RTF        Tier 2: accuracy, F1, deadline
        └──────────── Tier 3: Task Success Rate, Δ_ASR ───────────┘
```

**Run locally** (CPU, quick sanity check): open from `server/notebooks/`, kernel = `server/venv`.

**Run on Kaggle** (GPU, sweep many models):
1. Create a **private** Dataset containing `voice_eval.jsonl` + the `.m4a` files.
2. New Notebook → *File → Import notebook* → upload this file.
3. *Add Input* → your dataset. *Settings* → Accelerator **GPU T4**, **Internet on**.
4. *Run all*. Results land in `/kaggle/working/outputs/`.

⚠️ Quality metrics transfer between machines; **speed metrics (RTF, latency) do not** — Kaggle has a GPU, your server does not.

## 0. Config

In [ ]:
import glob
import json
import math
import os
import re
import shutil
import subprocess
import time
import unicodedata
from collections import Counter
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import requests

IS_KAGGLE = Path("/kaggle/input").exists()

if IS_KAGGLE:
    found = glob.glob("/kaggle/input/**/voice_eval.jsonl", recursive=True)
    assert found, "Add the private dataset that contains voice_eval.jsonl as notebook Input"
    DATA_DIR = Path(found[0]).parent
    AUDIO_DIR = Path("/kaggle/input")  # searched recursively
    OUTPUT_DIR = Path("/kaggle/working/outputs")
    ASR_MODELS = ["vinai/PhoWhisper-small", "vinai/PhoWhisper-medium", "vinai/PhoWhisper-large"]
    LLM_MODELS = ["qwen2.5:3b", "qwen2.5:7b", "gemma3:4b", "qwen3:4b"]
else:
    DATA_DIR = Path("data").resolve()
    AUDIO_DIR = Path(os.getenv("VOICE_AUDIO_DIR", "~/Downloads/Voice")).expanduser()
    OUTPUT_DIR = Path("outputs").resolve()
    ASR_MODELS = ["vinai/PhoWhisper-small"]
    LLM_MODELS = ["qwen2.5:3b"]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OLLAMA_URL = "http://localhost:11434"
print(f"Kaggle={IS_KAGGLE}\nDATA_DIR={DATA_DIR}\nAUDIO_DIR={AUDIO_DIR}\nASR={ASR_MODELS}\nLLM={LLM_MODELS}")

## 1. Setup: ffmpeg + Ollama (installs only on Kaggle)

In [ ]:
def run_shell(command: str) -> None:
    """Run a shell command and stop the notebook if it fails."""
    print("$", command)
    subprocess.run(command, shell=True, check=True)


if IS_KAGGLE:
    if shutil.which("ffmpeg") is None:
        run_shell("apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null")
    if shutil.which("ollama") is None:
        run_shell("apt-get -qq install -y zstd > /dev/null")  # the installer unpacks a .tar.zst
        run_shell("curl -fsSL https://ollama.com/install.sh | sh")

assert shutil.which("ffmpeg"), "ffmpeg is required to decode m4a"
assert shutil.which("ollama"), "Install Ollama first: https://docs.ollama.com"

In [ ]:
def is_ollama_running() -> bool:
    """Return True when the Ollama HTTP server answers."""
    try:
        return requests.get(f"{OLLAMA_URL}/api/tags", timeout=2).ok
    except requests.RequestException:
        return False


def start_ollama_server() -> None:
    """Start `ollama serve` in the background (if needed) and wait until it is ready."""
    if is_ollama_running():
        print("Ollama already running")
        return
    log_file = open(OUTPUT_DIR / "ollama.log", "w")
    subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=subprocess.STDOUT)
    for _ in range(60):
        if is_ollama_running():
            print("Ollama started")
            return
        time.sleep(1)
    raise RuntimeError(f"Ollama did not start, see {OUTPUT_DIR / 'ollama.log'}")


def pull_ollama_model(model_name: str) -> None:
    """Download an Ollama model (no-op if it is already present)."""
    start = time.perf_counter()
    response = requests.post(f"{OLLAMA_URL}/api/pull", json={"model": model_name, "stream": False}, timeout=3600)
    response.raise_for_status()
    print(f"pulled {model_name} in {time.perf_counter() - start:.0f}s")


start_ollama_server()
for llm_model in LLM_MODELS:
    pull_ollama_model(llm_model)

## 2. Data

One line of `voice_eval.jsonl` = one example. `audio: null` → text-only example (only the parser is evaluated).
`tags` let you slice results (e.g. `relative_date`, `no_deadline`).

In [ ]:
def load_eval_examples(path: Path) -> list[dict]:
    """Read the JSONL eval set, skipping blank lines."""
    with open(path, encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


def find_audio_file(file_name: str) -> Path | None:
    """Find an audio file by name anywhere under AUDIO_DIR."""
    matches = list(AUDIO_DIR.rglob(file_name))
    return matches[0] if matches else None


examples = load_eval_examples(DATA_DIR / "voice_eval.jsonl")
for example in examples:
    if example["audio"]:
        assert find_audio_file(example["audio"]), f"Missing audio file: {example['audio']}"

print(f"{len(examples)} examples, {sum(1 for e in examples if e['audio'])} with audio")
pd.DataFrame([{"id": e["id"], "audio": e["audio"], "text": e["transcript_gold"], "tags": ", ".join(e["tags"])}
              for e in examples])

## 3. Text normalization — why `"ộ" != "ộ"`

Vietnamese diacritics can be stored *composed* (NFC, 1 code point) or *decomposed* (NFD, base letter + combining marks).
They look identical but compare as different strings → WER would count them as errors. Always normalize to NFC first.

In [ ]:
def normalize_vietnamese_text(text: str) -> str:
    """NFC-normalize, lowercase and strip punctuation so transcripts can be compared fairly."""
    text = unicodedata.normalize("NFC", text).lower()
    text = re.sub(r"[^\w\s]", " ", text)
    return " ".join(text.split())


composed = "ộ"
decomposed = unicodedata.normalize("NFD", composed)
print("equal before normalizing:", composed == decomposed, "| code points:", len(composed), "vs", len(decomposed))
print("equal after normalizing: ", normalize_vietnamese_text(composed) == normalize_vietnamese_text(decomposed))
print(normalize_vietnamese_text("Ngày mai, 9 GIỜ nộp báo cáo!"))

## 4. Metrics — ✍️ YOUR TURN

Each function = one formula. Replace `raise NotImplementedError` with your implementation, then run the test cell below.
The pipeline (sections 5–7) still runs while metrics are TODO; their columns just show `TODO`.

Suggested order: `accuracy` → `real_time_factor` → `token_f1` → `keyword_recall` → `macro_f1` → `mean_absolute_error_hours` → `confidence_interval_95` → `word_error_rate` (the hardest, worth it).

In [ ]:
def accuracy(predictions: list, labels: list) -> float:
    """Fraction of exact matches.

    accuracy = (1/n) * Σ_i 1[ŷ_i == y_i]
    """
    raise NotImplementedError


def macro_f1(predictions: list, labels: list) -> float:
    """Unweighted mean of per-class F1 over every class seen in predictions or labels.

    P_c = TP_c / (TP_c + FP_c),  R_c = TP_c / (TP_c + FN_c)   (define 0/0 = 0)
    F1_c = 2·P_c·R_c / (P_c + R_c)
    macroF1 = (1/|C|) * Σ_c F1_c
    """
    raise NotImplementedError


def token_f1(predicted_text: str, reference_text: str) -> float:
    """Bag-of-tokens F1 between two normalized texts (tokens = Vietnamese syllables).

    overlap = |Counter(pred) ∩ Counter(ref)|
    P = overlap / |pred|,  R = overlap / |ref|,  F1 = 2PR / (P + R)
    Hint: use normalize_vietnamese_text() first.
    """
    raise NotImplementedError


def word_error_rate(hypothesis: str, reference: str) -> float:
    """Syllable-level WER via Levenshtein distance.

    WER = (S + D + I) / N,  N = number of reference syllables
    Hint: dynamic programming, d[i][j] = edit distance between ref[:i] and hyp[:j]
          d[i][j] = min(d[i-1][j] + 1,            # deletion
                        d[i][j-1] + 1,            # insertion
                        d[i-1][j-1] + (ref[i-1] != hyp[j-1]))  # substitution / match
    """
    raise NotImplementedError


def keyword_recall(transcript: str, keywords: list[str]) -> float:
    """Fraction of important keywords that appear in the transcript (after normalization).

    KR = #keywords found / #keywords
    """
    raise NotImplementedError


def real_time_factor(processing_seconds: float, audio_seconds: float) -> float:
    """How many seconds of compute per second of audio (< 1 means faster than real time).

    RTF = t_processing / t_audio
    """
    raise NotImplementedError


def mean_absolute_error_hours(predicted_times: list[datetime], true_times: list[datetime]) -> float:
    """Average absolute deadline error in hours.

    MAE_h = (1/m) * Σ_i |t̂_i − t_i| / 3600s
    """
    raise NotImplementedError


def confidence_interval_95(proportion: float, sample_size: int) -> tuple[float, float]:
    """Normal-approximation 95% confidence interval for a proportion.

    p ± 1.96 · sqrt(p(1 − p) / n)
    """
    raise NotImplementedError

In [ ]:
def check_metric(name: str, test) -> None:
    """Run one metric test and print ✅ / ❌ / ⬜ TODO."""
    try:
        test()
        print(f"✅ {name}")
    except NotImplementedError:
        print(f"⬜ {name}: TODO")
    except AssertionError as error:
        print(f"❌ {name}: {error}")


def close(a, b) -> bool:
    """Float comparison with a small tolerance."""
    return math.isclose(a, b, abs_tol=1e-4)


def test_accuracy():
    assert close(accuracy(["a", "b", "c"], ["a", "b", "d"]), 2 / 3), "expected 2/3"

def test_macro_f1():
    assert close(macro_f1(["a", "a", "b"], ["a", "b", "b"]), 2 / 3), "expected 2/3 (both classes F1 = 2/3)"
    assert close(macro_f1(["a", "a"], ["a", "a"]), 1.0), "perfect predictions → 1.0"

def test_token_f1():
    assert close(token_f1("Nộp bài tập lớn", "nộp bài tập lớn cấu trúc dữ liệu"), 8 / 11), "P=1, R=4/7 → 8/11"
    assert close(token_f1("abc", "xyz"), 0.0), "no overlap → 0"

def test_word_error_rate():
    assert close(word_error_rate("ngày mai chín giờ gian nộp", "ngày mai chín giờ sáng nộp"), 1 / 6), "1 substitution / 6"
    assert close(word_error_rate("a b", "a b c"), 1 / 3), "1 deletion / 3"
    assert close(word_error_rate("a x b c", "a b c"), 1 / 3), "1 insertion / 3"
    assert close(word_error_rate("Ngày MAI!", "ngày mai"), 0.0), "normalize first"

def test_keyword_recall():
    assert close(keyword_recall("tối mai 8 giờ ôn thi giải tích", ["tối mai", "thi", "Giải Tích", "vật lý"]), 3 / 4)

def test_real_time_factor():
    assert close(real_time_factor(14, 4), 3.5), "the CPU demo: 14s for 4s of audio"

def test_mean_absolute_error_hours():
    t = datetime(2026, 11, 10, 8, 0)
    assert close(mean_absolute_error_hours([t, t], [t + timedelta(hours=2), t]), 1.0)

def test_confidence_interval_95():
    low, high = confidence_interval_95(0.8, 50)
    assert close(low, 0.8 - 0.110873) and close(high, 0.8 + 0.110873), "0.8 ± 0.1109"


for metric_test in [test_accuracy, test_real_time_factor, test_token_f1, test_keyword_recall, test_macro_f1,
                    test_mean_absolute_error_hours, test_confidence_interval_95, test_word_error_rate]:
    check_metric(metric_test.__name__.removeprefix("test_"), metric_test)

## 5. Run ASR (PhoWhisper)

Results are cached in `outputs/asr_results.jsonl`, so re-running skips work already done.

In [ ]:
import torch
import transformers
from transformers import pipeline


def audio_duration_seconds(audio_path: Path) -> float:
    """Read audio length with ffprobe."""
    output = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(audio_path)],
        capture_output=True, text=True, check=True)
    return float(output.stdout.strip())


def load_jsonl_cache(path: Path) -> list[dict]:
    """Read cached result rows (empty list if the file does not exist yet)."""
    return load_eval_examples(path) if path.exists() else []


def append_jsonl(path: Path, row: dict) -> None:
    """Append one result row to a JSONL cache file."""
    with open(path, "a", encoding="utf-8") as file:
        file.write(json.dumps(row, ensure_ascii=False) + "\n")


class AsrRunner:
    """Holds one PhoWhisper model and transcribes audio files with it."""

    def __init__(self, model_name: str):
        """Load the model on GPU (fp16) when available, otherwise CPU (fp32)."""
        self.model_name = model_name
        use_gpu = torch.cuda.is_available()
        dtype = torch.float16 if use_gpu else torch.float32
        major_version = int(transformers.__version__.split(".")[0])
        dtype_argument = {"dtype": dtype} if major_version >= 5 else {"torch_dtype": dtype}
        self.pipe = pipeline("automatic-speech-recognition", model=model_name,
                             device=0 if use_gpu else "cpu", chunk_length_s=30, **dtype_argument)

    def transcribe(self, audio_path: Path) -> tuple[str, float]:
        """Return (text, processing seconds) for one audio file, forcing Vietnamese."""
        start = time.perf_counter()
        output = self.pipe(str(audio_path), generate_kwargs={"language": "vi", "task": "transcribe"})
        return output["text"].strip(), time.perf_counter() - start


ASR_CACHE = OUTPUT_DIR / "asr_results.jsonl"
audio_examples = [e for e in examples if e["audio"]]

for asr_model in ASR_MODELS:
    done_ids = {r["id"] for r in load_jsonl_cache(ASR_CACHE) if r["asr_model"] == asr_model}
    todo = [e for e in audio_examples if e["id"] not in done_ids]
    if not todo:
        print(f"{asr_model}: cached")
        continue
    runner = AsrRunner(asr_model)
    runner.transcribe(find_audio_file(todo[0]["audio"]))  # warm-up, not timed
    for example in todo:
        audio_path = find_audio_file(example["audio"])
        text, seconds = runner.transcribe(audio_path)
        append_jsonl(ASR_CACHE, {"asr_model": asr_model, "id": example["id"], "transcript": text,
                                 "processing_seconds": round(seconds, 3),
                                 "audio_seconds": round(audio_duration_seconds(audio_path), 3)})
        print(f"{asr_model} | {example['id']} | {text}")
    del runner
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

asr_results = load_jsonl_cache(ASR_CACHE)

## 6. Run parser (Ollama + JSON schema)

**Baseline**: the LLM computes `deadline_at` itself. Later experiment: let the LLM output *relative* time
(`weekday`, `day_offset`, `time`) and compute the date in Python — then compare the numbers.

Each LLM is run on two inputs: the **gold transcript** and every **ASR transcript** → lets us compute Δ_ASR.

In [ ]:
VIETNAMESE_WEEKDAYS = ["Thứ Hai", "Thứ Ba", "Thứ Tư", "Thứ Năm", "Thứ Sáu", "Thứ Bảy", "Chủ Nhật"]

TODO_DRAFT_SCHEMA = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},
        "category": {"type": "string", "enum": ["study", "assignment", "exam", "personal", "other"]},
        "priority": {"type": "string", "enum": ["low", "medium", "high"]},
        "deadline_at": {"type": ["string", "null"], "description": "ISO 8601 with +07:00 offset, or null"},
    },
    "required": ["title", "category", "priority", "deadline_at"],
}


def describe_reference_time(reference_time: str) -> str:
    """Turn an ISO timestamp into 'Thứ Ba, 2026-11-10 14:00 (+07:00)' for the prompt."""
    moment = datetime.fromisoformat(reference_time)
    return f"{VIETNAMESE_WEEKDAYS[moment.weekday()]}, {moment:%Y-%m-%d %H:%M} (+07:00)"


def build_system_prompt(reference_time: str) -> str:
    """System prompt telling the LLM how to turn one utterance into a todo."""
    return (
        "Bạn trích xuất MỘT việc cần làm từ câu nói tiếng Việt của sinh viên.\n"
        f"Thời điểm hiện tại: {describe_reference_time(reference_time)}.\n"
        "- title: ngắn gọn, viết hoa chữ đầu, không chứa thời gian.\n"
        "- category: study (học/ôn), assignment (bài tập/nộp/báo cáo), exam (thi/kiểm tra), personal, other.\n"
        "- priority: high nếu có 'gấp', 'quan trọng'; low nếu việc vặt; còn lại medium.\n"
        "- deadline_at: ISO 8601 +07:00. Có ngày mà không có giờ thì dùng 23:59. Không nhắc thời gian thì null."
    )


class OllamaTodoParser:
    """Turns a transcript into a todo_draft dict using an Ollama model with structured output."""

    def __init__(self, model_name: str):
        """Remember which Ollama model to call."""
        self.model_name = model_name

    def parse(self, transcript: str, reference_time: str) -> tuple[dict | None, float, str | None]:
        """Return (draft or None, seconds, error message or None)."""
        payload = {
            "model": self.model_name,
            "stream": False,
            "format": TODO_DRAFT_SCHEMA,
            "options": {"temperature": 0},
            "messages": [{"role": "system", "content": build_system_prompt(reference_time)},
                         {"role": "user", "content": transcript}],
        }
        if self.model_name.startswith("qwen3"):
            payload["think"] = False  # thinking mode is slow and not needed here
        start = time.perf_counter()
        try:
            response = requests.post(f"{OLLAMA_URL}/api/chat", json=payload, timeout=600)
            response.raise_for_status()
            draft = json.loads(response.json()["message"]["content"])
            return draft, time.perf_counter() - start, None
        except (requests.RequestException, json.JSONDecodeError, KeyError) as error:
            return None, time.perf_counter() - start, str(error)


def parser_inputs() -> list[tuple[str, dict, str]]:
    """List (source, example, transcript): gold text for all examples + every ASR output."""
    by_id = {e["id"]: e for e in examples}
    inputs = [("gold", e, e["transcript_gold"]) for e in examples]
    inputs += [(f"asr:{r['asr_model']}", by_id[r["id"]], r["transcript"]) for r in asr_results]
    return inputs


PARSER_CACHE = OUTPUT_DIR / "parser_results.jsonl"

for llm_model in LLM_MODELS:
    done = {(r["source"], r["id"]) for r in load_jsonl_cache(PARSER_CACHE) if r["llm_model"] == llm_model}
    todo = [(s, e, t) for s, e, t in parser_inputs() if (s, e["id"]) not in done]
    if not todo:
        print(f"{llm_model}: cached")
        continue
    parser = OllamaTodoParser(llm_model)
    parser.parse("test", examples[0]["reference_time"])  # warm-up: loads the model into memory
    for source, example, transcript in todo:
        draft, seconds, error = parser.parse(transcript, example["reference_time"])
        append_jsonl(PARSER_CACHE, {"llm_model": llm_model, "source": source, "id": example["id"],
                                    "transcript": transcript, "draft": draft,
                                    "processing_seconds": round(seconds, 3), "error": error})
        print(f"{llm_model} | {source:28} | {example['id']} | {json.dumps(draft, ensure_ascii=False)}")

parser_results = load_jsonl_cache(PARSER_CACHE)

## 7. Report

In [ ]:
def safe_metric(function, *args):
    """Call a metric, returning 'TODO' while it is not implemented yet."""
    try:
        return round(function(*args), 3)
    except NotImplementedError:
        return "TODO"


def same_deadline_day(predicted: str | None, expected: str | None) -> bool:
    """True when both deadlines are null, or both fall on the same calendar day."""
    if predicted is None or expected is None:
        return predicted is None and expected is None
    try:
        return datetime.fromisoformat(predicted).date() == datetime.fromisoformat(expected).date()
    except ValueError:
        return False


def is_task_success(draft: dict | None, expected: dict) -> bool:
    """Headline metric: category right AND deadline on the right day AND title roughly right."""
    if draft is None:
        return False
    try:
        title_ok = token_f1(draft.get("title", ""), expected["title"]) >= 0.5
    except NotImplementedError:
        title_ok = True  # until token_f1 exists, judge only category + deadline
    return (draft.get("category") == expected["category"]
            and same_deadline_day(draft.get("deadline_at"), expected["deadline_at"])
            and title_ok)


by_id = {e["id"]: e for e in examples}

# --- Tier 1: ASR ---
asr_rows = []
for asr_model in ASR_MODELS:
    rows = [r for r in asr_results if r["asr_model"] == asr_model]
    if not rows:
        continue
    asr_rows.append({
        "asr_model": asr_model, "n": len(rows),
        "WER": safe_metric(lambda: sum(word_error_rate(r["transcript"], by_id[r["id"]]["transcript_gold"])
                                       for r in rows) / len(rows)),
        "keyword_recall": safe_metric(lambda: sum(keyword_recall(r["transcript"], by_id[r["id"]]["keywords"])
                                                  for r in rows) / len(rows)),
        "RTF (this machine)": safe_metric(lambda: sum(real_time_factor(r["processing_seconds"], r["audio_seconds"])
                                                      for r in rows) / len(rows)),
    })
print("Tier 1 — ASR")
display(pd.DataFrame(asr_rows))

# --- Tier 2 + 3: parser and end-to-end ---
parser_rows = []
for (llm_model, source), group in pd.DataFrame(parser_results).groupby(["llm_model", "source"]):
    rows = group.to_dict("records")
    expected = [by_id[r["id"]]["expected"] for r in rows]
    drafts = [r["draft"] or {} for r in rows]
    successes = [is_task_success(r["draft"], exp) for r, exp in zip(rows, expected)]
    success_rate = sum(successes) / len(successes)
    with_deadline = [(d, e) for d, e in zip(drafts, expected) if e["deadline_at"] and d.get("deadline_at")]
    parser_rows.append({
        "llm_model": llm_model, "source": source, "n": len(rows),
        "schema_valid": round(sum(r["draft"] is not None for r in rows) / len(rows), 3),
        "category_acc": safe_metric(accuracy, [d.get("category") for d in drafts], [e["category"] for e in expected]),
        "category_macroF1": safe_metric(macro_f1, [d.get("category") for d in drafts], [e["category"] for e in expected]),
        "priority_acc": safe_metric(accuracy, [d.get("priority") for d in drafts], [e["priority"] for e in expected]),
        "deadline_day_acc": round(sum(same_deadline_day(d.get("deadline_at"), e["deadline_at"])
                                      for d, e in zip(drafts, expected)) / len(rows), 3),
        "deadline_MAE_h": safe_metric(lambda: mean_absolute_error_hours(
            [datetime.fromisoformat(d["deadline_at"]) for d, _ in with_deadline],
            [datetime.fromisoformat(e["deadline_at"]) for _, e in with_deadline])) if with_deadline else None,
        "title_tokenF1": safe_metric(lambda: sum(token_f1(d.get("title", ""), e["title"])
                                                 for d, e in zip(drafts, expected)) / len(rows)),
        "TASK_SUCCESS": round(success_rate, 3),
        "TSR_CI95": safe_metric(lambda: confidence_interval_95(success_rate, len(rows))[1] - success_rate),
        "latency_s (this machine)": round(group["processing_seconds"].mean(), 2),
    })
parser_table = pd.DataFrame(parser_rows)
print("Tier 2/3 — Parser & end-to-end   (TSR_CI95 = ± half-width)")
display(parser_table)

In [ ]:
# Δ_ASR: how much task success is lost because of ASR mistakes (audio examples only)
audio_ids = {e["id"] for e in audio_examples}
delta_rows = []
for llm_model in LLM_MODELS:
    rows = [r for r in parser_results if r["llm_model"] == llm_model and r["id"] in audio_ids]
    gold = [is_task_success(r["draft"], by_id[r["id"]]["expected"]) for r in rows if r["source"] == "gold"]
    for asr_model in ASR_MODELS:
        asr = [is_task_success(r["draft"], by_id[r["id"]]["expected"]) for r in rows if r["source"] == f"asr:{asr_model}"]
        if gold and asr:
            delta_rows.append({"llm_model": llm_model, "asr_model": asr_model,
                               "TSR_gold": sum(gold) / len(gold), "TSR_asr": sum(asr) / len(asr),
                               "Δ_ASR": sum(gold) / len(gold) - sum(asr) / len(asr)})
print("Δ_ASR = TSR(gold transcript) − TSR(ASR transcript)")
display(pd.DataFrame(delta_rows))

# Task success sliced by tag (gold transcripts) — shows WHERE each model fails
tag_rows = []
for r in parser_results:
    if r["source"] != "gold":
        continue
    for tag in by_id[r["id"]]["tags"]:
        tag_rows.append({"llm_model": r["llm_model"], "tag": tag,
                         "success": is_task_success(r["draft"], by_id[r["id"]]["expected"])})
print("Task success by tag (gold transcripts)")
display(pd.DataFrame(tag_rows).pivot_table(index="tag", columns="llm_model", values="success", aggfunc="mean").round(2))

In [ ]:
# Inspect failures one by one — reading errors teaches more than any average
failures = [
    {"llm": r["llm_model"], "source": r["source"], "id": r["id"], "transcript": r["transcript"],
     "pred_deadline": (r["draft"] or {}).get("deadline_at"), "exp_deadline": by_id[r["id"]]["expected"]["deadline_at"],
     "pred_cat": (r["draft"] or {}).get("category"), "exp_cat": by_id[r["id"]]["expected"]["category"],
     "pred_title": (r["draft"] or {}).get("title"), "error": r["error"]}
    for r in parser_results if not is_task_success(r["draft"], by_id[r["id"]]["expected"])
]
pd.set_option("display.max_colwidth", 60)
display(pd.DataFrame(failures))